<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">

# My First App for Monai label

This notebook will walk you through how to create your first app.

### Table of Contents
**[0.0 App Setup](#0.0-App-Setup)** <br>
**[1.0 Write Your App](#1.0-Write-Your-App)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.1 App Structure](#1.1-App-Structure) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.2 App for This Notebook](#1.2-App-for-This-Notebook) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.3 Config File](#1.3-Config-File) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.4 Write `infer/my_spleen`](#1.4-Write-infer/my_spleen) <br>
**[2.0 Open and Test with Slicer](#2.0-Open-and-Test-with-Slicer)** <br>
**[3.0 Start MONAI Label Server](#3.0-Start-MONAI-Label-Server)** <br>
**[4.0 My App Next Steps](#4.0-My-App-Next-Steps)** <br>

---
# 0.0 App Setup 
Let's start with some imports and variables that we will use later.

In [ ]:
import ipywidgets
import threading
import os
HOST_NAME=os.getenv('HOST_NAME')
HOST_IP=os.getenv('HOST_IP')
DLI_DATA_DIR=os.getenv('DLI_DATA_DIR')
os.chdir(DLI_DATA_DIR)
RADOLOGY_APP_ROOT=DLI_DATA_DIR+"/apps/radiology"


if len(HOST_NAME)==0:
    from requests import get
    HOST_NAME=get('https://api.ipify.org').text
    HOST_IP=HOST_NAME
print(f" HOST_NAME is {HOST_NAME} with IP {HOST_IP}")

---
# 1.0 Write Your App

## 1.1 App Structure 

A typical app structure is listed below. 
Each folder points to code to enable certain feature.
```bash
App
├── lib/infers is the module where researchers define the inference class (i.e. type of inferer, pre transforms for inference, etc).
├── lib/trainers is the module to define the pre and post transforms to train the network/model.
├── lib/configs is the module to define the image selection techniques.
├── lib/transforms is the module to define customised transformations to be used in the App.
├── lib/activelearning is the module to define the image selection techniques.
main.py is the script to extend MONAILabelApp class
```


## 1.2 App for This Notebook
We will be starting with the Radiology app and simply adding a couple of files to create our new model, `my_spleen`.

```bash
App
├── lib/infers 
├── lib/configs 
```

## 1.3 Config File
 
Run the cell below to write the file for `config/my_spleen`.

In [ ]:
%%writefile {RADOLOGY_APP_ROOT}/lib/configs/my_spleen.py
import logging
import os
from typing import Any, Dict, Optional, Union

from monai.networks.nets import UNet

from monailabel.interfaces.config import TaskConfig
from monailabel.interfaces.tasks.infer_v2 import InferTask
from monailabel.interfaces.tasks.scoring import ScoringMethod
from monailabel.interfaces.tasks.strategy import Strategy
from monailabel.interfaces.tasks.train import TrainTask
from monailabel.utils.others.generic import download_file, strtobool

logger = logging.getLogger(__name__)


class Myspleen(TaskConfig):
    def __init__(self):
        super().__init__()
    
        self.epistemic_enabled = None
        self.epistemic_samples = None
        self.tta_enabled = None
        self.tta_samples = None
    
    def init(self, name: str, model_dir: str, conf: Dict[str, str], planner: Any, **kwargs):
        super().init(name, model_dir, conf, planner, **kwargs)

        # Labels
        self.labels = {
            "my_spleen": 1,
        }

        # Model Files
        self.path = [
            # os.path.join(self.model_dir, f"pretrained_{name}.pt"),  # pretrained
            os.path.join(self.model_dir, f"segmentation_spleen.pt"),  # pretrained
            os.path.join(self.model_dir, f"{name}.pt"),  # published
        ]

        # Download PreTrained Model
        if strtobool(self.conf.get("use_pretrained_model", "true")):
            url = f"{self.conf.get('pretrained_path', self.PRE_TRAINED_PATH)}"
            url = f"{url}/radiology_segmentation_unet_spleen_total_seg.pt"
            download_file(url, self.path[0])

        # Network
        self.network = UNet(
            spatial_dims=3,
            in_channels=1,
            out_channels=2,
            channels=[16, 32, 64, 128, 256],
            strides=[2, 2, 2, 2],
            num_res_units=2,
            norm="batch",
        )
        logger.info(f"-------------------------------- in init ")

    def infer(self) -> Union[InferTask, Dict[str, InferTask]]:
        from lib.infers.my_spleen import MySpleen
        task: InferTask = MySpleen(
            path=self.path,
            network=self.network,
            labels=self.labels,
            preload=strtobool(self.conf.get("preload", "false")),
        )
        return task

    def trainer(self) -> Optional[TrainTask]:
        pass

    def strategy(self) -> Union[None, Strategy, Dict[str, Strategy]]:
        pass

    def scoring_method(self) -> Union[None, ScoringMethod, Dict[str, ScoringMethod]]:
        pass

## 1.4 Write `infer/my_spleen`

In [ ]:
%%writefile {RADOLOGY_APP_ROOT}/lib/infers/my_spleen.py

from typing import Callable, Sequence

from monai.inferers import Inferer, SlidingWindowInferer
from monai.transforms import (
    Activationsd,
    AsDiscreted,
    EnsureChannelFirstd,
    EnsureTyped,
    LoadImaged,
    ScaleIntensityRanged,
    Spacingd,
)

from monailabel.interfaces.tasks.infer import InferTask, InferType
from monailabel.transform.post import Restored


class MySpleen(InferTask):
    def __init__(
        self,
        path,
        network=None,
        type=InferType.SEGMENTATION,
        labels=None,
        dimension=3,
        description="A pre-trained model for volumetric (3D) segmentation of the spleen from CT image",
        **kwargs,
    ):
        super().__init__(
            path=path,
            network=network,
            type=type,
            labels=labels,
            dimension=dimension,
            description=description,
            **kwargs,
        )

    def pre_transforms(self, data=None) -> Sequence[Callable]:
        return [
            LoadImaged(keys="image"),
            EnsureTyped(keys="image", device=data.get("device") if data else None),
            EnsureChannelFirstd(keys="image"),
            Spacingd(keys="image", pixdim=[1.0, 1.0, 1.0]),
            ScaleIntensityRanged(keys="image", a_min=-157, a_max=164, b_min=0.0, b_max=1.0, clip=True),
        ]

    def inferer(self, data=None) -> Inferer:
        return SlidingWindowInferer(roi_size=(160, 160, 160))

    def post_transforms(self, data=None) -> Sequence[Callable]:
        return [
            EnsureTyped(keys="pred", device=data.get("device") if data else None),
            Activationsd(keys="pred", softmax=True),
            AsDiscreted(keys="pred", argmax=True),
            Restored(keys="pred", ref_image="image"),
        ]



---
# 2.0 Open and Test with Slicer
You will use Slicer, which was launched in the previous notebook.
<div class="alert alert-block alert-info">
Password:<b>gtc25</b>
</div>

In [ ]:
url_text='<h1><center><a href="http://'+HOST_NAME+'/x11/" target="_blank">'
url_text=url_text+'<img src="https://user-images.githubusercontent.com/15837524/103544638-6999fa80-4e6e-11eb-94c4-5c42cd2f63f8.png" width="200" height="100"/>'
url_text=url_text+'<br>Click to open 3D Slicer <br> in a new window</a></center></h1>'
ipywidgets.HTML(url_text)

---
# 3.0 Start MONAI Label Server
Now we need to start the MONAI Label server with our model config using the radiology app.

In [ ]:
%cd $DLI_DATA_DIR
!monailabel start_server --app apps/radiology \
            --studies datasets/Task09_Spleen/imagesTr \
            --conf models my_spleen

---
# 4.0 My App Next Steps
Experiment more with your new app using the following reference:
- [radiology app page](https://github.com/Project-MONAI/MONAILabel/tree/main/sample-apps/radiology#how-to-add-new-model)

<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">